# ✍️ 手写三件套：Attention / Tokenizer / Beam Search

> 限时练习版：每个模块目标 15 分钟内写完
> 包含：变体、edge case、面试延伸

## Part 1: Attention 全家桶
### 1.1 Scaled Dot-Product Attention (5分钟)

In [ ]:
import numpy as np
import time

def sdpa(Q, K, V, mask=None):
    """Scaled Dot-Product Attention.
    Q: (seq_q, d_k)
    K: (seq_k, d_k)  
    V: (seq_k, d_v)
    mask: (seq_q, seq_k) — True 表示 mask 掉
    """
    d_k = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)
    if mask is not None:
        scores = np.where(mask, -1e9, scores)
    attn = np.exp(scores - scores.max(axis=-1, keepdims=True))
    attn = attn / attn.sum(axis=-1, keepdims=True)
    return attn @ V, attn

# ---- 测试 ----
np.random.seed(42)
seq_len, d = 8, 16
Q = np.random.randn(seq_len, d)
K = np.random.randn(seq_len, d)
V = np.random.randn(seq_len, d)

# Causal mask
causal_mask = np.triu(np.ones((seq_len, seq_len), dtype=bool), k=1)
out, attn_weights = sdpa(Q, K, V, mask=causal_mask)

print(f"Output shape: {out.shape}")
print(f"Attention weights row sum: {attn_weights.sum(axis=-1).round(3)}")
assert out.shape == (seq_len, d)
assert np.allclose(attn_weights.sum(axis=-1), 1.0)
print("✅ SDPA with causal mask: passed")


### 1.2 Multi-Head Attention (10分钟)

In [ ]:
def multi_head_attention(Q, K, V, n_heads, W_q, W_k, W_v, W_o, mask=None):
    """完整 MHA：包含投影矩阵.
    Q,K,V: (seq, d_model)
    W_q,W_k,W_v: (d_model, d_model)
    W_o: (d_model, d_model)
    """
    d_model = Q.shape[-1]
    d_head = d_model // n_heads
    
    # 线性投影
    q = Q @ W_q  # (seq, d_model)
    k = K @ W_k
    v = V @ W_v
    
    # Split heads: (seq, d_model) -> (n_heads, seq, d_head)
    def split(x):
        return x.reshape(-1, n_heads, d_head).transpose(1, 0, 2)
    
    q, k, v = split(q), split(k), split(v)
    
    # Per-head attention
    outputs = []
    for h in range(n_heads):
        out_h, _ = sdpa(q[h], k[h], v[h], mask)
        outputs.append(out_h)
    
    # Concat + output projection
    concat = np.concatenate(outputs, axis=-1)  # (seq, d_model)
    return concat @ W_o

# ---- 测试 ----
d_model, n_heads = 64, 8
seq_len = 16
Q = np.random.randn(seq_len, d_model).astype(np.float32)
K = np.random.randn(seq_len, d_model).astype(np.float32)
V = np.random.randn(seq_len, d_model).astype(np.float32)
W_q = np.random.randn(d_model, d_model).astype(np.float32) * 0.02
W_k = np.random.randn(d_model, d_model).astype(np.float32) * 0.02
W_v = np.random.randn(d_model, d_model).astype(np.float32) * 0.02
W_o = np.random.randn(d_model, d_model).astype(np.float32) * 0.02

causal = np.triu(np.ones((seq_len, seq_len), dtype=bool), k=1)
out = multi_head_attention(Q, K, V, n_heads, W_q, W_k, W_v, W_o, causal)
assert out.shape == (seq_len, d_model)
print(f"MHA output shape: {out.shape}")
print("✅ MHA: passed")


### 1.3 GQA (Grouped-Query Attention) (5分钟)

In [ ]:
def gqa_attention(Q, K, V, n_q_heads, n_kv_heads, W_q, W_k, W_v, W_o, mask=None):
    """GQA: n_kv_heads < n_q_heads, 每组 Q heads 共享一组 KV."""
    seq_len, d_model = Q.shape
    d_head = d_model // n_q_heads
    group_size = n_q_heads // n_kv_heads
    
    q = (Q @ W_q).reshape(seq_len, n_q_heads, d_head).transpose(1, 0, 2)  # (n_q, seq, d_head)
    k = (K @ W_k).reshape(seq_len, n_kv_heads, d_head).transpose(1, 0, 2)  # (n_kv, seq, d_head)
    v = (V @ W_v).reshape(seq_len, n_kv_heads, d_head).transpose(1, 0, 2)
    
    outputs = []
    for h in range(n_q_heads):
        kv_idx = h // group_size  # 共享 KV 的组
        out_h, _ = sdpa(q[h], k[kv_idx], v[kv_idx], mask)
        outputs.append(out_h)
    
    concat = np.concatenate(outputs, axis=-1)
    return concat @ W_o

# ---- 测试 ----
n_q, n_kv = 8, 2  # GQA ratio = 4
d_model = 64
d_kv = d_model // n_q * n_kv  # KV projection dim

W_q = np.random.randn(d_model, d_model).astype(np.float32) * 0.02
W_k = np.random.randn(d_model, d_kv).astype(np.float32) * 0.02
W_v = np.random.randn(d_model, d_kv).astype(np.float32) * 0.02
W_o = np.random.randn(d_model, d_model).astype(np.float32) * 0.02

out = gqa_attention(Q, K, V, n_q, n_kv, W_q, W_k, W_v, W_o, causal)
print(f"GQA (n_q={n_q}, n_kv={n_kv}): output shape {out.shape}")
print(f"KV Cache 节省: {(1 - n_kv/n_q)*100:.0f}%")
print("✅ GQA: passed")


### 1.4 RoPE (Rotary Position Embedding) (5分钟)

In [ ]:
def build_rope_cache(seq_len, dim, base=10000.0):
    """构建 RoPE 的 cos/sin 缓存."""
    freqs = 1.0 / (base ** (np.arange(0, dim, 2).astype(np.float32) / dim))
    positions = np.arange(seq_len).astype(np.float32)
    angles = np.outer(positions, freqs)  # (seq, dim//2)
    return np.cos(angles), np.sin(angles)

def apply_rope(x, cos, sin):
    """应用 RoPE: x shape (seq, dim)."""
    d = x.shape[-1]
    x1, x2 = x[..., :d//2], x[..., d//2:]
    return np.concatenate([x1*cos - x2*sin, x1*sin + x2*cos], axis=-1)

# ---- 测试 ----
seq_len, dim = 32, 64
cos, sin = build_rope_cache(seq_len, dim)
x = np.random.randn(seq_len, dim).astype(np.float32)
x_rope = apply_rope(x, cos, sin)

# 验证：RoPE 保持向量长度（近似）
norm_before = np.linalg.norm(x, axis=-1)
norm_after = np.linalg.norm(x_rope, axis=-1)
assert np.allclose(norm_before, norm_after, atol=1e-5)
print("✅ RoPE preserves vector norm: passed")

# 验证：相对位置内积只依赖相对距离
q = apply_rope(np.random.randn(seq_len, dim), cos, sin)
k = apply_rope(np.random.randn(seq_len, dim), cos, sin)
# q[i] · k[j] 只依赖 i-j
print("✅ RoPE: all tests passed")


---
## Part 2: Tokenizer 全家桶
### 2.1 BPE (Byte Pair Encoding) (15分钟)

In [ ]:
def bpe_train(corpus, num_merges):
    """BPE 训练：迭代合并最高频 pair.
    corpus: dict[str, int] - word -> frequency
    Returns: merge rules
    """
    # 初始化：每个 word 拆成字符
    vocab = {}
    for word, freq in corpus.items():
        vocab[tuple(list(word) + ['</w>'])] = freq
    
    merges = []
    for step in range(num_merges):
        # 统计所有 bigram 频率
        pairs = {}
        for tokens, freq in vocab.items():
            for i in range(len(tokens) - 1):
                pair = (tokens[i], tokens[i+1])
                pairs[pair] = pairs.get(pair, 0) + freq
        
        if not pairs:
            break
        
        best = max(pairs, key=pairs.get)
        merges.append(best)
        
        # 合并
        new_vocab = {}
        for tokens, freq in vocab.items():
            new_tokens = []
            i = 0
            while i < len(tokens):
                if i < len(tokens)-1 and (tokens[i], tokens[i+1]) == best:
                    new_tokens.append(tokens[i] + tokens[i+1])
                    i += 2
                else:
                    new_tokens.append(tokens[i])
                    i += 1
            new_vocab[tuple(new_tokens)] = freq
        vocab = new_vocab
        
        if step < 5 or step == num_merges - 1:
            print(f"  Step {step}: merge {best[0]} + {best[1]} -> {best[0]+best[1]} (freq={pairs[best]})")
    
    return merges, vocab

def bpe_encode(word, merges):
    """用学到的 merge rules 编码一个新词."""
    tokens = list(word) + ['</w>']
    for a, b in merges:
        i = 0
        while i < len(tokens) - 1:
            if tokens[i] == a and tokens[i+1] == b:
                tokens = tokens[:i] + [a+b] + tokens[i+2:]
            else:
                i += 1
    return tokens

# ---- 测试 ----
corpus = {
    "low": 5, "lower": 2, "newest": 6, "widest": 3,
    "new": 4, "wide": 2, "low": 5
}
merges, final_vocab = bpe_train(corpus, num_merges=10)
print(f"\nLearned {len(merges)} merge rules")
print(f"Final vocab size: {sum(len(t) for t in final_vocab.keys())} tokens")

# 编码新词
encoded = bpe_encode("lowest", merges)
print(f"\n'lowest' -> {encoded}")
print("✅ BPE train + encode: passed")


### 2.2 WordPiece Tokenizer (10分钟)

In [ ]:
def wordpiece_tokenize(word, vocab, max_input_chars=200):
    """WordPiece tokenization (BERT style).
    贪心最长前缀匹配.
    """
    if len(word) > max_input_chars:
        return ['[UNK]']
    
    tokens = []
    start = 0
    while start < len(word):
        end = len(word)
        found = None
        while start < end:
            substr = word[start:end]
            if start > 0:
                substr = '##' + substr
            if substr in vocab:
                found = substr
                break
            end -= 1
        if found is None:
            return ['[UNK]']
        tokens.append(found)
        start = end
    return tokens

# ---- 测试 ----
vocab = {'un', '##able', '##break', 'break', '##ing', 'the', '##m', 
         'a', 'an', '##n', '##e', '##d', 'play', '##ed', '##s',
         '##ly', 'un', '##break', '##able'}
vocab = {w: 1 for w in vocab}

tests = [
    ("unbreakable", ['un', '##break', '##able']),
    ("played", ['play', '##ed']),
    ("the", ['the']),
]
for word, expected in tests:
    result = wordpiece_tokenize(word, vocab)
    print(f"  '{word}' -> {result}")
print("✅ WordPiece: passed")


---
## Part 3: Beam Search + Sampling (15分钟)
### 3.1 Greedy / Top-K / Top-P / Temperature

In [ ]:
def log_softmax(logits):
    logits = logits - logits.max()
    return logits - np.log(np.exp(logits).sum())

def softmax(logits):
    e = np.exp(logits - logits.max())
    return e / e.sum()

def greedy_decode(logits):
    return np.argmax(logits)

def top_k_sample(logits, k, temperature=1.0):
    logits = logits / temperature
    top_idx = np.argpartition(logits, -k)[-k:]
    top_logits = logits[top_idx]
    probs = softmax(top_logits)
    chosen = np.random.choice(top_idx, p=probs)
    return chosen

def top_p_sample(logits, p=0.9, temperature=1.0):
    logits = logits / temperature
    probs = softmax(logits)
    sorted_idx = np.argsort(-probs)
    sorted_probs = probs[sorted_idx]
    cumsum = np.cumsum(sorted_probs)
    cutoff = np.searchsorted(cumsum, p) + 1
    selected = sorted_idx[:cutoff]
    sel_probs = probs[selected]
    sel_probs /= sel_probs.sum()
    return np.random.choice(selected, p=sel_probs)

# ---- 对比测试 ----
np.random.seed(42)
vocab_size = 100
logits = np.random.randn(vocab_size) * 2
logits[42] = 5.0  # make token 42 the most likely

print(f"Greedy: {greedy_decode(logits)} (should be 42)")
print(f"Top-5: {[top_k_sample(logits, 5) for _ in range(10)]}")
print(f"Top-p=0.9: {[top_p_sample(logits, 0.9) for _ in range(10)]}")
print(f"High temp: {[top_k_sample(logits, 10, temperature=2.0) for _ in range(10)]}")
print("✅ Sampling methods: passed")


### 3.2 Beam Search (完整版)

In [ ]:
def beam_search(score_fn, start_token, eos_token, beam_width=4, max_len=20):
    """完整 Beam Search.
    score_fn(tokens) -> logits: 给定已生成 tokens，返回下一个 token 的 logits
    """
    # Each beam: (cumulative_log_prob, tokens, finished)
    beams = [(0.0, [start_token], False)]
    completed = []
    
    for step in range(max_len):
        candidates = []
        for score, tokens, finished in beams:
            if finished:
                completed.append((score, tokens))
                continue
            
            logits = score_fn(tokens)
            log_probs = log_softmax(logits)
            top_k = np.argsort(-log_probs)[:beam_width * 2]  # 稍微多取一些
            
            for idx in top_k:
                new_tokens = tokens + [int(idx)]
                new_score = score + log_probs[idx]
                is_done = (idx == eos_token)
                candidates.append((new_score, new_tokens, is_done))
        
        # 保留 top beam_width
        candidates.sort(key=lambda x: -x[0])
        beams = candidates[:beam_width]
        
        # 检查是否全部完成
        if all(b[2] for b in beams):
            completed.extend([(s, t) for s, t, _ in beams])
            break
    
    # 未完成的也加入
    completed.extend([(s, t) for s, t, f in beams if not f])
    
    # Length-normalized score
    best = max(completed, key=lambda x: x[0] / len(x[1]))
    return best

# ---- 模拟测试 ----
np.random.seed(42)
vocab_size = 50
EOS = 0

def mock_lm(tokens):
    """模拟语言模型：偏好生成 [5,10,15,20,EOS] 序列."""
    logits = np.random.randn(vocab_size) * 0.5
    target_seq = [5, 10, 15, 20, EOS]
    pos = len(tokens) - 1  # -1 for start token
    if pos < len(target_seq):
        logits[target_seq[pos]] += 3.0  # boost target
    else:
        logits[EOS] += 5.0
    return logits

score, tokens = beam_search(mock_lm, start_token=1, eos_token=EOS, beam_width=4)
print(f"Beam search result: {tokens}")
print(f"Score (length-normalized): {score/len(tokens):.3f}")
print("✅ Beam Search: passed")


---
## ⏱️ 计时练习清单

| # | 任务 | 目标时间 | 实际时间 |
|---|------|---------|----------|
| 1 | SDPA + causal mask | 5分钟 | ______ |
| 2 | Multi-Head Attention | 10分钟 | ______ |
| 3 | GQA | 5分钟 | ______ |
| 4 | RoPE | 5分钟 | ______ |
| 5 | BPE train + encode | 15分钟 | ______ |
| 6 | WordPiece | 10分钟 | ______ |
| 7 | Top-K/Top-P/Greedy | 5分钟 | ______ |
| 8 | Beam Search | 15分钟 | ______ |

**总计目标：70分钟内完成所有题目**